In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
PUPIL_ROOT = Path(r"C:\Users\cdd\Documents\Uni\Special_course\pupil_processed_clara")
EEG_ROOT   = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_processed2")
SUBJECTS   = [f"sub-{sid:03d}" for sid in range(32, 99) if sid not in (37,66,94)]

In [3]:
from pathlib import Path

# Base directory
base_dir = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_processed2")

missing = []

# Iterate over all subdirectories matching "sub-*"
for sub in base_dir.glob("sub-*"):
    memory_dir = sub / "memory" / "05"
    target = memory_dir / "epoch_035.csv"

    if not target.exists():
        missing.append(str(sub.name))

if missing:
    print("The following subjects are missing epoch_035.csv in their memory folder:")
    for subj in missing:
        print(f"  • {subj}")
else:
    print("All memory folders contain epoch_035.csv.")


All memory folders contain epoch_035.csv.


subject 32 and 61 should not have them (at least they dont for pupillometry). This is a bit weird, for now i am going to remove them from the analysis. Subject 42, indeed has extra trials in pupillometry and EEG, we keep it.
Now lets see how we skip the rejected trials in pupillometry from EEG too.

In [4]:
"""
trial_order_from_fif.py  —  robustly recover trial order
========================================================
Problem fixed: when triggers are stored as **annotations**, the numeric event
codes get remapped to arbitrary indices by `mne.events_from_annotations`, so our
previous parser never saw the real codes.  The new version:

1. **Inspects `raw.annotations` directly.**  Each annotation’s *description* is
   the 7-digit trigger (e.g. "6001051").
2. Falls back to the stim channel if annotations are absent.
3. Keeps only triggers whose *position field* is `01` (trial start).
4. Outputs `trial_order.tsv` with
   `subject  condition  load  trial_idx` (chronological `trial_idx` per subject).

Run:
```bash
python trial_order_from_fif.py
```

Dependencies: `mne >= 1.0`, `pandas`, `pathlib` (built-in).
"""

from pathlib import Path
from typing import List
import pandas as pd
import mne

# ─────────────────────────────────────────────────────────────────────────────
BASE = Path(r"C:\Users\cdd\Documents\Uni\Special_course")
EEG_FIF_ROOT = BASE / "code" / "Special_course" / "EEG_amica_processed"
OUTFILE      = BASE / "trial_order_eeg.tsv"
# ─────────────────────────────────────────────────────────────────────────────


def decode_event_code(code_str: str):
    """Decode *code_str* (7-digit string) → (condition, load, position)."""
    if len(code_str) > 8  or not code_str.isdigit():
        print(f"{code_str} is not a 7-digit or 6 number")
        raise ValueError("code_str must be a number")
        

    s = code_str
    cond_code, pos_code, len_code = s[:2], s[2:4], s[4:6]

    condition = {"50": "control", "60": "memory"}.get(cond_code)
    if condition is None:
        raise ValueError("condition code not 50/60")

    load = {"05": 5, "09": 9, "13": 13}.get(len_code)
    if load is None:
        raise ValueError("unexpected seq length")

    position = int(pos_code)
    return condition, load, position


def extract_trials_from_annotations(raw, subject: str) -> List[tuple]:
    rows = []
    trial_idx = 0
    for onset, duration, desc in zip(raw.annotations.onset,
                                     raw.annotations.duration,
                                     raw.annotations.description):
        try:
            cond, load, pos = decode_event_code(desc)
        except ValueError:
            continue
        if pos != 1:
            continue
        rows.append((subject, cond, load, trial_idx))
        trial_idx += 1
    return rows


def extract_trials_from_stim(raw, subject: str) -> List[tuple]:
    rows = []
    trial_idx = 0
    events = mne.find_events(raw, verbose="ERROR")
    for _, _, code in events:
        try:
            cond, load, pos = decode_event_code(str(code))
        except ValueError:
            continue
        if pos != 1:
            continue
        rows.append((subject, cond, load, trial_idx))
        trial_idx += 1
    return rows


def trial_order_for_subject(fif_path: Path) -> pd.DataFrame:
    subj = fif_path.parent.name
    raw = mne.io.read_raw_fif(fif_path, preload=False, verbose="ERROR")

    rows = (extract_trials_from_annotations(raw, subj)
            if raw.annotations else [])
    if not rows:
        rows = extract_trials_from_stim(raw, subj)
    if not rows:
        raise RuntimeError("no trial-start events found")

    return pd.DataFrame(rows, columns=["subject", "condition", "load", "trial_idx"])


def build_table() -> pd.DataFrame:
    dfs = []
    for fif in sorted(EEG_FIF_ROOT.glob("sub-*/*.fif")):
        try:
            dfs.append(trial_order_for_subject(fif))
        except Exception as e:
            print(f"‼️  {fif.name}: {e}")
    if not dfs:
        raise RuntimeError("no subjects processed successfully")
    return pd.concat(dfs, ignore_index=True)


def main():
    table = build_table()
    table.to_csv(OUTFILE, sep="\t", index=False)
    print(f"Wrote trial order for {table.subject.nunique()} subjects → {OUTFILE}")


if __name__ == "__main__":
    main()


boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is not a 7-digit or 6 number
boundary is 

In [7]:
# ───────────────────────────────────────────────────────────────────────────
BASE = Path(r"C:\Users\cdd\Documents\Uni\Special_course")
TSV_ROOT   = BASE / "ds003838-download"
OUTFILE    = BASE / "trial_order_pupil.tsv"
# ───────────────────────────────────────────────────────────────────────────


def decode_label(label: str):
    """Return (condition, load, position) parsed from a numeric *label* string."""
    if not label.isdigit() or len(label) < 6:
        raise ValueError("label isn’t a 6+/7‑digit number")

    cond_code = label[:2]
    position  = int(label[2:4])
    load_code = label[4:6]

    condition = {"50": "control", "60": "memory"}.get(cond_code)
    if condition is None:
        raise ValueError("unknown condition code")

    load = {"05": 5, "09": 9, "13": 13}.get(load_code)
    if load is None:
        raise ValueError("unexpected load code")

    return condition, load, position


def trial_order_for_subject(tsv_path: Path) -> pd.DataFrame:
    subject = tsv_path.parent.parent.name  # sub-XXX
    df      = pd.read_csv(tsv_path, sep="\t")

    rows = []
    trial_idx = 0

    for label in df["label"]:
        try:
            cond, load, pos = decode_label(str(label))
        except ValueError:
            continue
        if pos != 1:
            continue
        rows.append((subject, cond, load, trial_idx))
        trial_idx += 1

    if not rows:
        raise RuntimeError(f"no trial‑start codes found in {tsv_path}")

    return pd.DataFrame(rows, columns=["subject", "condition", "load", "trial_idx"])


def build_table():
    all_rows = []
    pattern = TSV_ROOT.glob("sub-*/pupil/*_task-memory_events.tsv")
    for tsv in sorted(pattern):
        try:
            all_rows.append(trial_order_for_subject(tsv))
        except Exception as e:
            print(f"‼️  {tsv.name}: {e}")
    if not all_rows:
        raise RuntimeError("no TSV files processed successfully")
    return pd.concat(all_rows, ignore_index=True)


if __name__ == "__main__":
    table = build_table()
    table.to_csv(OUTFILE, sep="\t", index=False)
    print(f"Wrote trial order for {table.subject.nunique()} subjects → {OUTFILE}")


Wrote trial order for 64 subjects → C:\Users\cdd\Documents\Uni\Special_course\trial_order_pupil.tsv


In [12]:
from pathlib import Path
import pandas as pd
import sys

# Base data directory
BASE = Path(r"C:\Users\cdd\Documents\Uni\Special_course")

EEG_ORDER_FILE  = BASE / "trial_order_eeg.tsv"
TSV_ORDER_FILE  = BASE / "trial_order_pupil.tsv"


def _fmt_sub(sub):
    """Return canonical subject string (e.g. 32 → sub-032)."""
    sub = str(sub)
    if sub.startswith("sub-"):
        return sub
    return f"sub-{int(sub):03d}"


def compare_trial_order(subject: str,
                        eeg_file: Path = EEG_ORDER_FILE,
                        tsv_file: Path = TSV_ORDER_FILE,
                        verbose: bool = True) -> bool:
    """Return True if the two trial‑order tables match exactly for *subject*.

    *Matches* means:
    1. Both tables contain the same number of trials for this subject.
    2. After sorting by `trial_idx`, every corresponding row shares the same
       `(condition, load)`.
    """
    sub = _fmt_sub(subject)

    try:
        df_eeg = pd.read_csv(eeg_file, sep="\t")
        df_tsv = pd.read_csv(tsv_file, sep="\t")
    except Exception as e:
        raise RuntimeError(f"Error loading order files: {e}")

    e1 = df_eeg[df_eeg["subject"] == sub].sort_values("trial_idx")
    e2 = df_tsv[df_tsv["subject"] == sub].sort_values("trial_idx")

    if e1.empty or e2.empty:
        if verbose:
            print(f"‼️  Subject {sub} not found in one of the tables.")
        return False

    if len(e1) != len(e2):
        if verbose:
            print(f"❌ Trial counts differ — EEG: {len(e1)}, TSV: {len(e2)}")
        return False

    mism = (e1[["condition", "load"]].reset_index(drop=True) !=
            e2[["condition", "load"]].reset_index(drop=True)).any(axis=1)

    if mism.any():
        if verbose:
            bad_rows = mism[mism].index.tolist()[:5]  # show up to 5 mismatches
            print(f"❌ Order mismatch at trial indices {bad_rows} (showing up to 5).")
        return False

    if verbose:
        print(f"✅ Trial order matches for {sub} (n={len(e1)} trials).")
    return True


# Compare trial order for all subjects from 032 to 098
mismatches = []
matches = []

for sid in range(32, 99):
    # Skip known missing subjects based on cell 3
    if sid in [37, 66, 94]:
        continue
        
    subject = f"sub-{sid:03d}"
    # First check without verbose output
    if compare_trial_order(subject, verbose=False):
        matches.append(subject)
    else:
        # Run again with verbose=True to see the mismatch details
        compare_trial_order(subject, verbose=True)
        mismatches.append(subject)

print(f"\nSummary:")
print(f"Matches: {len(matches)} subjects")
print(f"Mismatches: {len(mismatches)} subjects")
if mismatches:
    print(f"Mismatched subjects: {', '.join(mismatches)}")

❌ Trial counts differ — EEG: 162, TSV: 144
❌ Trial counts differ — EEG: 162, TSV: 137
‼️  Subject sub-096 not found in one of the tables.

Summary:
Matches: 61 subjects
Mismatches: 3 subjects
Mismatched subjects: sub-032, sub-061, sub-096


In [29]:
from __future__ import annotations
from pathlib import Path
import argparse
import pandas as pd
from typing import Dict, Tuple, List

# ---------------------------------------------------------------------------
#  Epoch‑range specification  (start, end) are **inclusive**
# ---------------------------------------------------------------------------
RANGES_DEFAULT: Dict[Tuple[str, int], Tuple[int, int]] = {
    ("memory", 5):   (0,   35),
    ("control", 5):  (36,  53),
    ("memory", 9):   (54,  89),
    ("control", 9):  (90,  107),
    ("memory", 13):  (108, 143),
    ("control", 13): (144, 161),
}

RANGES_SUB42: Dict[Tuple[str, int], Tuple[int, int]] = {
    ("memory", 5):   (0,   35),
    ("control", 5):  (36,  54),
    ("memory", 9):   (55,  90),
    ("control", 9):  (91,  109),
    ("memory", 13):  (110, 145),
    ("control", 13): (146, 164),
}

# ---------------------------------------------------------------------------
#  Core logic
# ---------------------------------------------------------------------------

def _build_counters(ranges: Dict[Tuple[str, int], Tuple[int, int]]):
    """Return mutable counters {key → next_epoch_int}."""
    return {k: v[0] for k, v in ranges.items()}


def _next_epoch(counters: Dict[Tuple[str, int], int], key: Tuple[str, int], ranges):
    start, end = ranges[key]
    cur = counters[key]
    if cur > end:
        raise ValueError(f"Out of epoch numbers for {key}; exceeded {end}.")
    counters[key] += 1
    return cur


def add_epoch_column(df: pd.DataFrame) -> pd.DataFrame:
    """Append `epoch_num` respecting per‑subject rules and return the new DF."""
    df = df.copy().sort_values(["subject", "trial_idx"])  # ensure order
    epoch_nums: List[int] = []

    for sub, sub_df in df.groupby("subject", sort=False):
        ranges = RANGES_SUB42 if sub == "sub-042" else RANGES_DEFAULT
        ctrs = _build_counters(ranges)
        for cond, load in zip(sub_df["condition"], sub_df["load"]):
            epoch_nums.append(_next_epoch(ctrs, (cond, int(load)), ranges))

    df["epoch_num"] = epoch_nums
    return df

def add_epoch_corrected_column(
        in_tsv: str | Path,
        out_tsv: str | Path | None = None,
) -> pd.DataFrame:
    """
    Load the TSV that already contains `epoch_num`, append a new
    column `epoch_corrected`, and (optionally) write it back.

    `epoch_corrected` is simply a sequential counter (00, 01, …)
    **inside every (subject, condition, load) folder**.
    """
    df = pd.read_csv(in_tsv, sep="\t")

    # Sequential index per folder
    df["epoch_corrected"] = (
        df.groupby(["subject", "condition", "load"])
          .cumcount()
          .map(lambda x: f"{x:02d}")          # zero-padded 2-digit string
    )

    # Save if the user provided an output path
    if out_tsv is not None:
        Path(out_tsv).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_tsv, sep="\t", index=False)

    return df
    

# ---------------------------------------------------------------------------
#  CLI entry‑point
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    in_path = r"C:\Users\cdd\Documents\Uni\Special_course\trial_order_eeg.tsv"
    out_path = r"C:\Users\cdd\Documents\Uni\Special_course\trial_order_eeg.tsv_with_epoch.tsv"

    df_in = pd.read_csv(in_path, sep="\t")
    df_out = add_epoch_column(df_in)
    df_out.to_csv(out_path, sep="\t", index=False)
    print(f"✅  Saved {len(df_out)} rows to {out_path}")


✅  Saved 10209 rows to C:\Users\cdd\Documents\Uni\Special_course\trial_order_eeg.tsv_with_epoch.tsv


In [30]:
add_epoch_corrected_column(out_path, out_path)


,subject,condition,load,trial_idx,epoch_num,epoch_corrected
0,sub-032,control,13,0,144,00
1,sub-032,control,5,1,36,00
2,sub-032,control,9,2,90,00
3,sub-032,memory,5,3,0,00
4,sub-032,memory,9,4,54,00
...,...,...,...,...,...,...
10204,sub-098,memory,9,157,89,35
10205,sub-098,memory,5,158,35,35
10206,sub-098,control,5,159,53,17
10207,sub-098,control,13,160,161,17


In [ ]:
# ----------------------------------------------------------------------
from pathlib import Path
from typing  import Union
import pandas as pd

PathLike = Union[str, Path]

SKIP_LOG = (
    r"C:\Users\cdd\Documents\Uni\Special_course\pupil_processed_clara"
    r"\trial_skip_log.tsv"
)

def add_epoch_corrected_skip_column(
        in_tsv : PathLike,                    # table that ALREADY has epoch_corrected
        out_tsv: PathLike | None = None,
        skip_log: PathLike = SKIP_LOG,
) -> pd.DataFrame:
    """
    • Copy `epoch_corrected` into `epoch_corrected_skip`.
    • Wherever (sub,cond,load,trial_idx) appears in the pupil skip-log,
      set that row to <NA>.
    • Keep counting 00, 01, 02 … only on the *kept* rows.

    That yields exactly: 00, 01, NA, 02, 03 … inside every folder.
    """
    # ---------------- 1. load the two tables ----------------
    df   = pd.read_csv(in_tsv,  sep="\t")
    skip = pd.read_csv(skip_log, sep="\t",
                       usecols=["subject","condition","load","trial_idx"])

    # ---------------- 2. normalise key columns --------------
    for col in ("subject", "condition"):
        df[col]   = df[col].astype(str).str.strip().str.lower()
        skip[col] = skip[col].astype(str).str.strip().str.lower()

    for col in ("load", "trial_idx"):
        df[col]   = df[col].astype(int)
        skip[col] = skip[col].astype(int)

    # -------- 3. build a fast lookup: {(sub,cond,load): {trial_idx,…}} ----
    skip_lookup: dict[tuple[str,str,int], set[int]] = {}
    for rec in skip.itertuples(index=False):
        key = (rec.subject, rec.condition, rec.load)
        skip_lookup.setdefault(key, set()).add(rec.trial_idx)

    # ---------------- 4. compute epoch_corrected_skip ---------
    def folder_logic(folder: pd.DataFrame) -> pd.Series:
        key        = (folder["subject"].iat[0],
                      folder["condition"].iat[0],
                      folder["load"].iat[0])
        bad_trials = skip_lookup.get(key, set())

        counter = 0
        out = []
        for tidx in folder["trial_idx"]:
            if tidx in bad_trials:
                out.append(pd.NA)
            else:
                out.append(f"{counter:02d}")
                counter += 1
        return pd.Series(out, index=folder.index)

    df["epoch_corrected_skip"] = (
        df.sort_values(["subject","condition","load","trial_idx"])
          .groupby(["subject","condition","load"], group_keys=False)
          .apply(folder_logic)
    )

    # --------------- 5. save / return -----------------------
    if out_tsv:
        Path(out_tsv).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_tsv, sep="\t", index=False)

    return df
# ----------------------------------------------------------------------


In [32]:
add_epoch_corrected_skip_column(
    r"C:\Users\cdd\Documents\Uni\Special_course\trial_order_eeg.tsv_with_epoch.tsv",
    r"C:\Users\cdd\Documents\Uni\Special_course\trial_order_eeg.tsv_with_epoch_skip.tsv"
)

C:\Users\cdd\AppData\Local\Temp\ipykernel_33620\3116280062.py:64: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.sort_values(["subject","condition","load","trial_idx"])


,subject,condition,load,trial_idx,epoch_num,epoch_corrected,epoch_corrected_skip
0,sub-032,control,13,0,144,0,00
1,sub-032,control,5,1,36,0,00
2,sub-032,control,9,2,90,0,00
3,sub-032,memory,5,3,0,0,00
4,sub-032,memory,9,4,54,0,00
...,...,...,...,...,...,...,...
10204,sub-098,memory,9,157,89,35,12
10205,sub-098,memory,5,158,35,35,<NA>
10206,sub-098,control,5,159,53,17,<NA>
10207,sub-098,control,13,160,161,17,<NA>


In [5]:
from pathlib import Path
import pandas as pd
import sys

# --------------------------------------------------------------------
# EDIT these three paths if your layout changes
MAPPING_TSV = (
    r"C:\Users\cdd\Documents\Uni\Special_course"
    r"\trial_order_eeg.tsv_with_epoch_skip.tsv"
)
EEG_ROOT = (
    r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course"
    r"\eeg_theta_processed2"
)
# --------------------------------------------------------------------

def rename_epochs(subject: str) -> None:
    """
    Rename files like
        …\sub-032\control\05\epoch_036.csv
    ▶︎ …\sub-032\control\05\epoch_001.csv
    where the mapping is read from the master TSV.

    Only rows for this subject are touched; other subjects stay intact.
    """
    subj = subject if subject.startswith("sub-") else f"sub-{int(subject):03d}"

    # --- read mapping for this subject only -------------------------
    cols = ["subject", "condition", "load", "trial_idx", "epoch_num"]
    df = (pd.read_csv(MAPPING_TSV, sep="\t", usecols=cols)
            .query("subject == @subj"))

    if df.empty:
        print(f"No rows for {subj} in mapping file.")
        return

    # --- go through every row and rename ----------------------------
    for row in df.itertuples(index=False):
        folder = (
            Path(EEG_ROOT) / row.subject / row.condition
            / f"{int(row.load):02d}"
        )
        old_f = folder / f"epoch_{int(row.epoch_num):03d}.csv"
        new_f = folder / f"trial_{int(row.trial_idx):03d}.csv"

        if not old_f.exists():
            print(f"⚠️  missing: {old_f}")
            continue

        if new_f.exists():
            print(f"⚠️  target already exists, skipping: {new_f}")
            continue

        old_f.rename(new_f)
        print(f"{old_f.name}  →  {new_f.name}")

    print(f"\nDone renaming files for {subj}.")

# Loop through all subjects from 033 to 099
for sid in range(32, 99):
    # Skip known missing subjects
    if sid in [37, 66, 94]:
        continue
    
    subject = f"sub-{sid:03d}"
    print(f"\nProcessing {subject}...")
    rename_epochs(subject)
    
print("\nAll subjects processed.")


Processing sub-032...
epoch_144.csv  →  trial_000.csv
epoch_036.csv  →  trial_001.csv
epoch_090.csv  →  trial_002.csv
epoch_000.csv  →  trial_003.csv
epoch_054.csv  →  trial_004.csv
epoch_108.csv  →  trial_005.csv
epoch_001.csv  →  trial_006.csv
epoch_055.csv  →  trial_007.csv
epoch_056.csv  →  trial_008.csv
epoch_002.csv  →  trial_009.csv
epoch_109.csv  →  trial_010.csv
epoch_110.csv  →  trial_011.csv
epoch_111.csv  →  trial_012.csv
epoch_057.csv  →  trial_013.csv
epoch_003.csv  →  trial_014.csv
epoch_091.csv  →  trial_015.csv
epoch_037.csv  →  trial_016.csv
epoch_145.csv  →  trial_017.csv
epoch_038.csv  →  trial_018.csv
epoch_092.csv  →  trial_019.csv
epoch_146.csv  →  trial_020.csv
epoch_058.csv  →  trial_021.csv
epoch_004.csv  →  trial_022.csv
epoch_112.csv  →  trial_023.csv
epoch_113.csv  →  trial_024.csv
epoch_114.csv  →  trial_025.csv
epoch_059.csv  →  trial_026.csv
epoch_115.csv  →  trial_027.csv
epoch_060.csv  →  trial_028.csv
epoch_005.csv  →  trial_029.csv
epoch_061.csv  → 

In [6]:
from pathlib import Path
import pandas as pd
import sys, re

# ── EDIT these if you ever move the files ────────────────────────────
MAPPING_TSV = (
    r"C:\Users\cdd\Documents\Uni\Special_course"
    r"\trial_order_eeg.tsv_with_epoch_skip.tsv"
)
PUPIL_ROOT = (
    r"C:\Users\cdd\Documents\Uni\Special_course\pupil_processed_clara"
)
# ─────────────────────────────────────────────────────────────────────

TMP_PATTERN = "__tmp_old{old:03d}_new{new:03d}.csv"

def rename_pupil_epochs(subject: str) -> None:
    """Rename pupil epoch_###.csv → trial_###.csv for one subject."""
    subj = subject if subject.startswith("sub-") else f"sub-{int(subject):03d}"

    # ---------- 1. pull mapping rows for this subject ----------------
    cols = ["subject", "condition", "load",
            "epoch_corrected_skip", "trial_idx"]
    df = (pd.read_csv(MAPPING_TSV, sep="\t", usecols=cols)
            .query("subject == @subj & epoch_corrected_skip.notna()"))

    if df.empty:
        print(f"No mapping rows for {subj}.")
        return

    # ---------- 2. two-phase rename to avoid collisions --------------
    for (cond, load), rows in df.groupby(["condition", "load"]):
        folder = Path(PUPIL_ROOT) / subj / cond / f"{int(load):02d}"
        if not folder.exists():
            print(f"⚠️  Missing folder: {folder}")
            continue

        # Phase A – temp names
        temp_pairs = []           # (tmp_path, final_path) per row
        for r in rows.itertuples(index=False):
            old_no = int(r.epoch_corrected_skip)
            new_no = int(r.trial_idx)

            old_f = folder / f"epoch_{old_no:03d}.csv"
            final_f = folder / f"trial_{new_no:03d}.csv"
            if not old_f.exists():
                print(f"⚠️  missing: {old_f}")
                continue

            tmp_f = folder / TMP_PATTERN.format(old=old_no, new=new_no)
            old_f.rename(tmp_f)
            temp_pairs.append((tmp_f, final_f))

        # Phase B – temp → final
        for tmp_f, final_f in temp_pairs:
            if final_f.exists():
                print(f"⚠️  {final_f.name} already exists, keeping temp.")
                continue
            tmp_f.rename(final_f)
            print(f"{tmp_f.name}  →  {final_f.name}")

    print(f"\nFinished renaming pupil files for {subj}.")


# Loop through all subjects from 32 to 99
for sid in range(32, 99):
    # Skip known missing subjects
    if sid in [37, 66, 94]:
        continue
    
    subject = f"sub-{sid:03d}"
    print(f"\nProcessing {subject}...")
    rename_pupil_epochs(subject)
    
print("\nAll subjects processed.")



Processing sub-032...
⚠️  missing: C:\Users\cdd\Documents\Uni\Special_course\pupil_processed_clara\sub-032\control\05\epoch_015.csv
⚠️  missing: C:\Users\cdd\Documents\Uni\Special_course\pupil_processed_clara\sub-032\control\05\epoch_016.csv
__tmp_old000_new001.csv  →  trial_001.csv
__tmp_old001_new016.csv  →  trial_016.csv
__tmp_old002_new018.csv  →  trial_018.csv
__tmp_old003_new034.csv  →  trial_034.csv
__tmp_old004_new037.csv  →  trial_037.csv
__tmp_old005_new053.csv  →  trial_053.csv
__tmp_old006_new069.csv  →  trial_069.csv
__tmp_old007_new072.csv  →  trial_072.csv
__tmp_old008_new088.csv  →  trial_088.csv
__tmp_old009_new092.csv  →  trial_092.csv
__tmp_old010_new106.csv  →  trial_106.csv
__tmp_old011_new109.csv  →  trial_109.csv
__tmp_old012_new123.csv  →  trial_123.csv
__tmp_old013_new127.csv  →  trial_127.csv
__tmp_old014_new142.csv  →  trial_142.csv
⚠️  missing: C:\Users\cdd\Documents\Uni\Special_course\pupil_processed_clara\sub-032\control\09\epoch_014.csv
⚠️  missing: C:\U

In [3]:
# match sampling rates if eeg is 250Hz and pupil is 100Hz
from scipy.signal import resample_poly
SUBJECTS = np.setdiff1d(np.arange(32, 99), [33, 34, 37, 66, 94])

FS_IN   = 250        # original sampling rate
FS_OUT  = 100        # target sampling rate
UP      = 2          # polyphase factors  (250 → 100 = 2/5)
DOWN    = 5

EEG_ROOT = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_hilbert")


for sub in SUBJECTS:
    sub_tag = f"sub-{sub:03d}"
    eeg_sub = EEG_ROOT / sub_tag
    if not eeg_sub.exists():
        print(f"{sub_tag}: EEG folder missing - skipped")
        continue
    print(f"Processing {sub_tag}...")

    for cond_path in eeg_sub.iterdir():          # memory / control
        for load_path in cond_path.iterdir():    # 05 / 09 / 13
            for trial_file in load_path.glob("trial_*.csv"):
                # Read the EEG epoch file (comma-separated)
                df = pd.read_csv(trial_file, comment="#") 

                time = df.iloc[:, 0].to_numpy(float)        # or df['time'].to_numpy(float)
                eeg  = df.iloc[:, 1:].to_numpy()            # shape: (n_samples, n_channels)

                # ------------------------------------------------------------------
                # 1. resample 250 → 100 Hz with anti-alias
                # ------------------------------------------------------------------
                eeg_ds = resample_poly(eeg, up=UP, down=DOWN, axis=0)

                # resample the time column
                n_out   = eeg_ds.shape[0]
                time_ds = np.arange(n_out) / FS_OUT + time[0]
                time_ds = np.round(time_ds, 2)

                # ------------------------------------------------------------------
                # 2. save back (overwrite or new file)
                # ------------------------------------------------------------------
                df_out = pd.DataFrame(
                    np.column_stack([time_ds, eeg_ds]),
                    columns=df.columns
                )
                df_out.to_csv(trial_file, index=False)
                print(f"Saved {trial_file}  (shape: {df_out.shape})")

            

Processing sub-032...
Saved C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_hilbert\sub-032\control\05\trial_001.csv  (shape: (656, 18))
Saved C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_hilbert\sub-032\control\05\trial_016.csv  (shape: (656, 18))
Saved C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_hilbert\sub-032\control\05\trial_018.csv  (shape: (656, 18))
Saved C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_hilbert\sub-032\control\05\trial_034.csv  (shape: (657, 18))
Saved C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_hilbert\sub-032\control\05\trial_037.csv  (shape: (658, 18))
Saved C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_hilbert\sub-032\control\05\trial_053.csv  (shape: (657, 18))
Saved C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_hilbert\sub-032\control\05\trial_056.csv  (shape: (657, 18))
Saved